# Xiangqi Lab — DQN

Set up the board rules, train DQN on a fixed endgame, and export a model for the website.

## 1. Load the project rules

Loads the project rules and checks the included source files.

In [ ]:
# @title Load the shared environment (run this cell first)
import hashlib
import importlib
import json
from pathlib import Path
import random
import sys
import tempfile

SOURCE_FILES = json.loads('{"rules.py": "\\"\\"\\"Dependency-free Xiangqi rules and a reusable, turn-based experiment state.\\n\\nCoordinates: board[row][col], row 0 is Black\'s home rank; UCI a0 is Red\'s.\\nResearch draw policy: threefold position repetition / 120 non-capture plies.\\nThis is not the full competition perpetual-check/chase adjudicator.\\n\\"\\"\\"\\nfrom collections import Counter\\nimport copy\\nimport re\\n\\nSTART_FEN = \\"rheakaehr/9/1c5c1/p1p1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RHEAKAEHR w - - 0 1\\"\\nPIECES = \\"KAEHRCPkaehrcp\\"\\nPRESETS = [\\n    {\\"id\\": \\"rook\\", \\"name\\": \\"Rook & pawn\\", \\"tag\\": \\"Starter · 6 pieces\\", \\"description\\": \\"Coordinate a rook and an advancing pawn.\\", \\"fen\\": \\"4k4/4a4/9/9/4p4/2P6/9/4R4/9/3K5 w - - 0 1\\"},\\n    {\\"id\\": \\"cannon\\", \\"name\\": \\"Cannon coordination\\", \\"tag\\": \\"Explore · 8 pieces\\", \\"description\\": \\"Explore cannon screens and pawn coordination.\\", \\"fen\\": \\"3aka3/9/9/4p4/9/4P4/2C6/9/9/3AK4 w - - 0 1\\"},\\n    {\\"id\\": \\"horse\\", \\"name\\": \\"Horse & pawn\\", \\"tag\\": \\"Explore · 8 pieces\\", \\"description\\": \\"Control horse paths and key squares.\\", \\"fen\\": \\"3ak4/9/4e4/9/4p4/2P6/5H3/9/9/3AK4 w - - 0 1\\"},\\n    {\\"id\\": \\"start\\", \\"name\\": \\"Starting position\\", \\"tag\\": \\"Reference · 32 pieces\\", \\"description\\": \\"A full board for demonstrations and rule checks.\\", \\"fen\\": START_FEN},\\n]\\n\\ndef color(piece):\\n    return \\"red\\" if piece.isupper() else \\"black\\"\\n\\ndef other(side):\\n    return \\"black\\" if side == \\"red\\" else \\"red\\"\\n\\ndef square(row, col):\\n    return chr(97 + col) + str(9 - row)\\n\\ndef coords(name):\\n    if not re.fullmatch(r\\"[a-i][0-9]\\", name):\\n        raise ValueError(\\"Coordinates must be between a0 and i9.\\")\\n    return 9 - int(name[1]), ord(name[0]) - 97\\n\\ndef action_id(move):\\n    r1, c1 = coords(move[:2]); r2, c2 = coords(move[2:])\\n    return (r1 * 9 + c1) * 90 + r2 * 9 + c2\\n\\ndef action_move(action):\\n    if type(action) is not int or not 0 <= action < 8100:\\n        raise ValueError(\\"Action IDs must be between 0 and 8099.\\")\\n    src, dst = divmod(action, 90)\\n    return square(*divmod(src, 9)) + square(*divmod(dst, 9))\\n\\ndef inside(r, c):\\n    return 0 <= r < 10 and 0 <= c < 9\\n\\ndef palace(r, c, side):\\n    return 3 <= c <= 5 and (7 <= r <= 9 if side == \\"red\\" else 0 <= r <= 2)\\n\\ndef pseudo(board, r, c, tr, tc):\\n    piece = board[r][c]\\n    if not piece or (r, c) == (tr, tc) or not inside(tr, tc):\\n        return False\\n    target = board[tr][tc]\\n    side = color(piece)\\n    if target and color(target) == side:\\n        return False\\n    dr, dc = tr-r, tc-c\\n    ar, ac = abs(dr), abs(dc)\\n    kind = piece.lower()\\n    blockers = -1\\n    if r == tr:\\n        blockers = sum(bool(board[r][i]) for i in range(min(c, tc)+1, max(c, tc)))\\n    elif c == tc:\\n        blockers = sum(bool(board[i][c]) for i in range(min(r, tr)+1, max(r, tr)))\\n    if kind == \\"r\\":\\n        return blockers == 0\\n    if kind == \\"c\\":\\n        return blockers == (1 if target else 0)\\n    if kind == \\"h\\":\\n        if (ar, ac) == (2, 1):\\n            return not board[r + dr//2][c]\\n        if (ar, ac) == (1, 2):\\n            return not board[r][c + dc//2]\\n        return False\\n    if kind == \\"e\\":\\n        return ar == ac == 2 and (tr >= 5 if side == \\"red\\" else tr <= 4) and not board[r+dr//2][c+dc//2]\\n    if kind == \\"a\\":\\n        return ar == ac == 1 and palace(tr, tc, side)\\n    if kind == \\"k\\":\\n        return (ar+ac == 1 and palace(tr, tc, side)) or (target and target.lower() == \\"k\\" and c == tc and blockers == 0)\\n    if kind == \\"p\\":\\n        return (dr == (-1 if side == \\"red\\" else 1) and dc == 0) or (dr == 0 and ac == 1 and (r <= 4 if side == \\"red\\" else r >= 5))\\n    return False\\n\\ndef in_check(board, side):\\n    king = \\"K\\" if side == \\"red\\" else \\"k\\"\\n    pos = next(((r,c) for r in range(10) for c in range(9) if board[r][c] == king), None)\\n    if pos is None:\\n        return True\\n    return any(p and color(p) != side and pseudo(board,r,c,*pos) for r,row in enumerate(board) for c,p in enumerate(row))\\n\\ndef legal_moves(board, side):\\n    moves = []\\n    for r,row in enumerate(board):\\n        for c,p in enumerate(row):\\n            if not p or color(p) != side:\\n                continue\\n            for tr in range(10):\\n                for tc in range(9):\\n                    target = board[tr][tc]\\n                    if target.lower() == \\"k\\" or not pseudo(board,r,c,tr,tc):\\n                        continue\\n                    board[r][c], board[tr][tc] = \\"\\", p\\n                    safe = not in_check(board,side)\\n                    board[r][c], board[tr][tc] = p, target\\n                    if safe:\\n                        moves.append(square(r,c)+square(tr,tc))\\n    return moves\\n\\ndef parse_fen(fen):\\n    if not isinstance(fen,str) or len(fen) > 250:\\n        raise ValueError(\\"Invalid FEN format.\\")\\n    fields = fen.strip().split()\\n    if len(fields) not in (2,6) or fields[1] not in (\\"w\\",\\"r\\",\\"b\\"):\\n        raise ValueError(\\"FEN requires a board and side w/r/b, optionally followed by - - halfmove fullmove.\\")\\n    rows = fields[0].replace(\\"n\\",\\"h\\").replace(\\"N\\",\\"H\\").replace(\\"b\\",\\"e\\").replace(\\"B\\",\\"E\\").split(\\"/\\")\\n    if len(rows) != 10:\\n        raise ValueError(\\"FEN must contain 10 ranks.\\")\\n    board = []\\n    for row in rows:\\n        result = []\\n        for ch in row:\\n            if ch in \\"123456789\\": result.extend([\\"\\"] * int(ch))\\n            elif ch in PIECES: result.append(ch)\\n            else: raise ValueError(\\"FEN contains an unknown piece.\\")\\n        if len(result) != 9: raise ValueError(\\"Each FEN rank must contain 9 squares.\\")\\n        board.append(result)\\n    side = \\"black\\" if fields[1] == \\"b\\" else \\"red\\"\\n    half, full = 0, 1\\n    if len(fields) == 6:\\n        if fields[2:4] != [\\"-\\",\\"-\\"]: raise ValueError(\\"FEN fields 3 and 4 must be - -.\\")\\n        try: half, full = int(fields[4]), int(fields[5])\\n        except ValueError: raise ValueError(\\"FEN move counters must be integers.\\")\\n        if not 0 <= half <= 100000 or not 1 <= full <= 100000: raise ValueError(\\"Move counters are out of range.\\")\\n    counts = Counter(p for row in board for p in row if p)\\n    for p,limit in zip(\\"KAEHRCP\\",[1,2,2,2,2,2,5]):\\n        if counts[p] > limit or counts[p.lower()] > limit: raise ValueError(\\"Piece counts exceed standard Xiangqi limits.\\")\\n    if counts[\\"K\\"] != 1 or counts[\\"k\\"] != 1: raise ValueError(\\"Each side must have exactly one king.\\")\\n    for r,row in enumerate(board):\\n        for c,p in enumerate(row):\\n            if not p: continue\\n            s = color(p)\\n            if p.lower() in \\"ka\\" and not palace(r,c,s): raise ValueError(\\"Kings and advisors must be inside their own palace.\\")\\n            if p.lower() == \\"a\\" and (r,c) not in ([(9,3),(9,5),(8,4),(7,3),(7,5)] if s == \\"red\\" else [(0,3),(0,5),(1,4),(2,3),(2,5)]): raise ValueError(\\"An advisor is on an invalid square.\\")\\n            if p.lower() == \\"e\\" and (r,c) not in ([(9,2),(9,6),(7,0),(7,4),(7,8),(5,2),(5,6)] if s == \\"red\\" else [(0,2),(0,6),(2,0),(2,4),(2,8),(4,2),(4,6)]): raise ValueError(\\"An elephant is on an invalid square.\\")\\n            if p.lower() == \\"p\\" and ((s == \\"red\\" and (r > 6 or (r >= 5 and c%2))) or (s == \\"black\\" and (r < 3 or (r <= 4 and c%2)))): raise ValueError(\\"A pawn is on an unreachable square.\\")\\n    if in_check(board, other(side)):\\n        raise ValueError(\\"The non-moving side is in check or the kings face each other. Adjust the position or side to move.\\")\\n    return board, side, half, full\\n\\ndef make_fen(board, side, half=0, full=1):\\n    rows=[]\\n    for row in board:\\n        out=\\"\\"; empty=0\\n        for p in row:\\n            if not p: empty+=1\\n            else:\\n                if empty: out+=str(empty); empty=0\\n                out+=p\\n        if empty: out+=str(empty)\\n        rows.append(out)\\n    return \\"/\\".join(rows)+f\\" {\'w\' if side == \'red\' else \'b\'} - - {half} {full}\\"\\n\\nclass Game:\\n    def __init__(self, fen=START_FEN, max_plies=160):\\n        self.board, self.side, self.halfmove, self.fullmove = parse_fen(fen)\\n        self.initial_fen = self.fen\\n        self.max_plies = max_plies\\n        self.history=[]\\n        self.repetitions=Counter({self.key:1})\\n        self._legal=None\\n\\n    @property\\n    def fen(self): return make_fen(self.board,self.side,self.halfmove,self.fullmove)\\n    @property\\n    def key(self): return \\" \\".join(self.fen.split()[:2])\\n    @property\\n    def legal(self):\\n        if self._legal is None: self._legal=legal_moves(self.board,self.side)\\n        return self._legal\\n\\n    def outcome(self):\\n        if not self.legal:\\n            return {\\"winner\\":other(self.side), \\"reason\\":\\"checkmate\\" if in_check(self.board,self.side) else \\"stalemate\\", \\"terminated\\":True, \\"truncated\\":False}\\n        if self.repetitions[self.key] >= 3: reason=\\"repetition\\"\\n        elif self.halfmove >= 120: reason=\\"no_capture\\"\\n        elif len(self.history) >= self.max_plies:\\n            return {\\"winner\\":None,\\"reason\\":\\"ply_limit\\",\\"terminated\\":False,\\"truncated\\":True}\\n        else: return None\\n        return {\\"winner\\":None,\\"reason\\":reason,\\"terminated\\":True,\\"truncated\\":False}\\n\\n    def step(self, move, actor=\\"human\\", analysis=None):\\n        if self.outcome(): raise ValueError(\\"The episode has ended. Reset or load another position.\\")\\n        if move not in self.legal: raise ValueError(\\"Illegal move: you cannot leave your king in check or expose facing kings.\\")\\n        r,c=coords(move[:2]); tr,tc=coords(move[2:])\\n        entry={\\"move\\":move,\\"side\\":self.side,\\"piece\\":self.board[r][c],\\"captured\\":self.board[tr][tc],\\"fen_before\\":self.fen,\\"actor\\":actor,\\"analysis\\":analysis or {}}\\n        self.board[tr][tc], self.board[r][c] = self.board[r][c], \\"\\"\\n        self.halfmove=0 if entry[\\"captured\\"] else self.halfmove+1\\n        if self.side == \\"black\\": self.fullmove+=1\\n        self.side=other(self.side); self._legal=None\\n        self.repetitions[self.key]+=1\\n        entry[\\"fen_after\\"]=self.fen\\n        self.history.append(entry)\\n        return self.outcome()\\n\\n    def undo(self):\\n        if not self.history: return\\n        self.repetitions[self.key]-=1\\n        entry=self.history.pop()\\n        self.board,self.side,self.halfmove,self.fullmove=parse_fen(entry[\\"fen_before\\"])\\n        self._legal=None\\n\\n    def observation(self):\\n        planes=[[[int(p == kind) for p in row] for row in self.board] for kind in PIECES]\\n        return {\\"fen\\":self.fen,\\"board_planes\\":planes,\\"plane_order\\":PIECES,\\"side_to_move\\":self.side,\\"legal_moves\\":list(self.legal),\\"legal_action_ids\\":[action_id(m) for m in self.legal],\\"halfmove_clock\\":self.halfmove,\\"ply\\":len(self.history),\\"max_plies\\":self.max_plies,\\"repetition_counts\\":dict(self.repetitions)}\\n\\n    def snapshot(self):\\n        return {\\"fen\\":self.fen,\\"initial_fen\\":self.initial_fen,\\"board\\":copy.deepcopy(self.board),\\"side\\":self.side,\\"legal_moves\\":list(self.legal),\\"in_check\\":in_check(self.board,self.side),\\"history\\":copy.deepcopy(self.history),\\"outcome\\":self.outcome(),\\"halfmove_clock\\":self.halfmove,\\"max_plies\\":self.max_plies,\\"piece_count\\":sum(bool(p) for row in self.board for p in row)}\\n\\n    def export(self):\\n        return {\\"schema\\":\\"xiangqi-lab.episode.v1\\",\\"rules\\":\\"xiangqi-research-v1\\",\\"draw_policy\\":\\"threefold-position-or-120-noncapture-plies\\",\\"initial_fen\\":self.initial_fen,\\"max_plies\\":self.max_plies,\\"moves\\":copy.deepcopy(self.history),\\"final_fen\\":self.fen,\\"outcome\\":self.outcome()}\\n", "training.py": "\\"\\"\\"Turn-based two-player environment; no ML framework required.\\n\\nreset -> (observation, info); step -> (observation, reward, terminated, truncated, info)\\nReward is from the ACTING side\'s perspective, not always Red\'s.\\nThis API resembles Gymnasium but is not a gymnasium.Env subclass.\\n\\"\\"\\"\\nfrom rules import Game, START_FEN, action_move\\n\\nclass XiangqiEnv:\\n    def __init__(self, max_plies=160):\\n        self.max_plies=max_plies\\n        self.game=None\\n\\n    def reset(self, *, fen=START_FEN, seed=None):\\n        self.game=Game(fen,self.max_plies)\\n        return self.game.observation(), {\\"outcome\\":self.game.outcome(),\\"seed\\":seed}\\n\\n    def step(self, action):\\n        if self.game is None: raise RuntimeError(\\"Call reset before step.\\")\\n        move=action_move(action) if type(action) is int else action\\n        side=self.game.side\\n        result=self.game.step(move,actor=\\"training\\")\\n        reward=0.0\\n        if result and result[\\"winner\\"]:\\n            reward=1.0 if result[\\"winner\\"] == side else -1.0\\n        return self.game.observation(), reward, bool(result and result[\\"terminated\\"]), bool(result and result[\\"truncated\\"]), {\\"acting_side\\":side,\\"outcome\\":result,\\"rewards\\":{\\"red\\": (1 if result and result[\\"winner\\"] == \\"red\\" else -1 if result and result[\\"winner\\"] == \\"black\\" else 0),\\"black\\": (1 if result and result[\\"winner\\"] == \\"black\\" else -1 if result and result[\\"winner\\"] == \\"red\\" else 0)}}\\n", "notebook_tools.py": "\\"\\"\\"Small, standard-library helpers for the environment notebook (no trainer).\\"\\"\\"\\nfrom html import escape\\n\\nfrom rules import Game, PRESETS\\n\\n\\nENDGAMES = {p[\\"id\\"]: dict(p) for p in PRESETS if p[\\"id\\"] in (\\"rook\\", \\"cannon\\", \\"horse\\")}\\nPIECE_LABELS = dict(zip(\\"KAEHRCPkaehrcp\\", \\"帥仕相馬車炮兵將士象馬車砲卒\\"))\\n\\n\\ndef action_mask(env):\\n    \\"\\"\\"An 8100-entry mask; an ended episode has no available actions.\\"\\"\\"\\n    if env.game is None:\\n        raise RuntimeError(\\"Call reset before asking for legal actions.\\")\\n    mask = [False] * 8100\\n    if env.game.outcome() is None:\\n        for index in env.game.observation()[\\"legal_action_ids\\"]:\\n            mask[index] = True\\n    return mask\\n\\n\\ndef replay_episode(episode):\\n    \\"\\"\\"Replay the website\'s JSON format, validating every move and final FEN.\\"\\"\\"\\n    if not isinstance(episode, dict) or episode.get(\\"schema\\") != \\"xiangqi-lab.episode.v1\\":\\n        raise ValueError(\\"Unsupported episode format.\\")\\n    moves = episode.get(\\"moves\\", [])\\n    limit = episode.get(\\"max_plies\\", 160)\\n    if not isinstance(moves, list) or len(moves) > 1000:\\n        raise ValueError(\\"Invalid move list.\\")\\n    if type(limit) is not int or not 2 <= limit <= 1000:\\n        raise ValueError(\\"The ply limit must be between 2 and 1000.\\")\\n    game = Game(episode.get(\\"initial_fen\\", \\"\\"), limit)\\n    for entry in moves:\\n        if not isinstance(entry, dict):\\n            raise ValueError(\\"Invalid move record.\\")\\n        game.step(entry.get(\\"move\\", \\"\\"), actor=str(entry.get(\\"actor\\", \\"import\\"))[:100])\\n    if episode.get(\\"final_fen\\") and game.fen != episode[\\"final_fen\\"]:\\n        raise ValueError(\\"The move history does not match the final FEN.\\")\\n    return game\\n\\n\\ndef board_svg(game, title=\\"Xiangqi Lab\\"):\\n    \\"\\"\\"Browser-rendered SVG: Red at the bottom, traditional piece symbols.\\"\\"\\"\\n    title = escape(str(title))\\n    parts = [\\n        \'<svg xmlns=\\"http://www.w3.org/2000/svg\\" viewBox=\\"0 0 620 744\\" \'\\n        \'width=\\"620\\" role=\\"img\\" aria-label=\\"Xiangqi board\\">\',\\n        f\'<title>{title}</title>\',\\n        \'<rect width=\\"620\\" height=\\"744\\" rx=\\"18\\" fill=\\"#f6f1e8\\"/>\',\\n        \'<g font-family=\\"Arial, sans-serif\\" fill=\\"#29423e\\">\',\\n        f\'<text x=\\"40\\" y=\\"38\\" font-size=\\"22\\" font-weight=\\"bold\\">{title}</text>\',\\n        f\'<text x=\\"40\\" y=\\"65\\" font-size=\\"15\\">Turn: {game.side.title()} · Move count: {len(game.history)}</text>\',\\n        \'</g>\',\\n        \'<rect x=\\"32\\" y=\\"85\\" width=\\"556\\" height=\\"612\\" rx=\\"10\\" fill=\\"#efe1c7\\"/>\',\\n        \'<g stroke=\\"#a68b61\\" stroke-width=\\"1.4\\">\',\\n    ]\\n\\n    def line(x1, y1, x2, y2):\\n        parts.append(f\'<line x1=\\"{x1}\\" y1=\\"{y1}\\" x2=\\"{x2}\\" y2=\\"{y2}\\"/>\')\\n\\n    for row in range(10):\\n        line(70, 120 + row * 60, 550, 120 + row * 60)\\n    for col in range(9):\\n        x = 70 + col * 60\\n        if col in (0, 8):\\n            line(x, 120, x, 660)\\n        else:\\n            line(x, 120, x, 360)\\n            line(x, 420, x, 660)\\n    for y in (120, 540):\\n        line(250, y, 370, y + 120)\\n        line(370, y, 250, y + 120)\\n    parts += [\'</g>\', \'<g font-family=\\"Arial, sans-serif\\" text-anchor=\\"middle\\" fill=\\"#806a48\\">\',\\n              \'<text x=\\"310\\" y=\\"397\\" font-size=\\"18\\" letter-spacing=\\"6\\">RIVER</text>\']\\n    for col in range(9):\\n        parts.append(f\'<text x=\\"{70 + col * 60}\\" y=\\"688\\" font-size=\\"13\\">{chr(97 + col)}</text>\')\\n    for row in range(10):\\n        parts.append(f\'<text x=\\"20\\" y=\\"{125 + row * 60}\\" font-size=\\"13\\">{9 - row}</text>\')\\n    parts.append(\'</g>\')\\n    for row, cells in enumerate(game.board):\\n        for col, piece in enumerate(cells):\\n            if not piece:\\n                continue\\n            x, y = 70 + col * 60, 120 + row * 60\\n            ink = \\"#ab3e2f\\" if piece.isupper() else \\"#29423e\\"\\n            parts.extend([\\n                f\'<g><title>{piece} on {chr(97 + col)}{9 - row}</title>\',\\n                f\'<circle cx=\\"{x}\\" cy=\\"{y}\\" r=\\"25\\" fill=\\"#fff6dc\\" stroke=\\"{ink}\\" stroke-width=\\"1.4\\"/>\',\\n                f\'<circle cx=\\"{x}\\" cy=\\"{y}\\" r=\\"21.5\\" fill=\\"none\\" stroke=\\"#d7c49e\\"/>\',\\n                f\'<text x=\\"{x}\\" y=\\"{y + 11}\\" text-anchor=\\"middle\\" font-size=\\"32\\" \'\\n                f\'font-family=\\"Noto Serif CJK SC, Microsoft YaHei, PingFang SC, serif\\" fill=\\"{ink}\\">{PIECE_LABELS[piece]}</text></g>\',\\n            ])\\n    result = game.outcome()\\n    footer = \\"Game in progress\\" if result is None else f\\"End: {result[\'reason\']} · Winner: {result[\'winner\'] or \'none\'}\\"\\n    parts.append(f\'<text x=\\"40\\" y=\\"726\\" font-size=\\"15\\" font-family=\\"Arial, sans-serif\\" fill=\\"#29423e\\">{escape(footer)}</text></svg>\')\\n    return \\"\\\\n\\".join(parts)\\n", "dqn_model.py": "\\"\\"\\"DQN network and stateless website policy; PyTorch is an optional dependency.\\"\\"\\"\\nfrom functools import lru_cache\\n\\nimport torch\\nfrom torch import nn\\n\\nMODEL_SCHEMA = \\"xiangqi-lab.dqn.v1\\"\\nPIECE_ORDER = \\"KAEHRCPkaehrcp\\"\\nACTION_COUNT = 8100\\n\\n\\n@lru_cache(maxsize=32768)\\ndef position_tokens(key):\\n    \\"\\"\\"Encode a position key without changing orientation or action numbering.\\"\\"\\"\\n    board, side = key.split()[:2]\\n    tokens = []\\n    for char in board:\\n        if char == \\"/\\":\\n            continue\\n        if char in \\"123456789\\":\\n            tokens.extend([0] * int(char))\\n        else:\\n            tokens.append(PIECE_ORDER.index(char) + 1)\\n    if len(tokens) != 90 or side not in (\\"w\\", \\"b\\", \\"r\\"):\\n        raise ValueError(\\"Invalid position key.\\")\\n    return bytes(tokens), int(side == \\"b\\")\\n\\n\\ndef encode_state(observation):\\n    \\"\\"\\"Keep every repetition entry, not only the current position\'s count.\\n\\n    Values are standard Python types so replay can be saved safely with torch.\\n    The external episode time limit is not an input to the continuing-game value.\\n    \\"\\"\\"\\n    key = \\" \\".join(observation[\\"fen\\"].split()[:2])\\n    board, side = position_tokens(key)\\n    history = tuple((*position_tokens(k), int(count))\\n                    for k, count in sorted(observation[\\"repetition_counts\\"].items()) if count > 0)\\n    if not history:\\n        raise ValueError(\\"The observation must include repetition counts.\\")\\n    return board, side, float(observation[\\"halfmove_clock\\"]) / 120.0, history\\n\\n\\ndef batch_states(states, device):\\n    boards, metadata, history_boards, history_meta, owners = [], [], [], [], []\\n    for owner, (board, side, clock, history) in enumerate(states):\\n        boards.append(list(board))\\n        metadata.append([side, clock, len(history) / 160.0])\\n        for old_board, old_side, count in history:\\n            history_boards.append(list(old_board))\\n            history_meta.append([old_side, count / 3.0])\\n            owners.append(owner)\\n    return (\\n        torch.tensor(boards, dtype=torch.long, device=device),\\n        torch.tensor(metadata, dtype=torch.float32, device=device),\\n        torch.tensor(history_boards, dtype=torch.long, device=device),\\n        torch.tensor(history_meta, dtype=torch.float32, device=device),\\n        torch.tensor(owners, dtype=torch.long, device=device),\\n    )\\n\\n\\nclass QNetwork(nn.Module):\\n    \\"\\"\\"Score all action IDs, using the current board and the full repetition map.\\"\\"\\"\\n    def __init__(self, hidden=64, embedding=4, history_hidden=16):\\n        super().__init__()\\n        self.config = {\\"hidden\\": hidden, \\"embedding\\": embedding, \\"history_hidden\\": history_hidden}\\n        self.pieces = nn.Embedding(15, embedding, padding_idx=0)\\n        self.current = nn.Sequential(nn.Linear(90 * embedding + 3, hidden), nn.ReLU())\\n        self.history = nn.Sequential(nn.Linear(90 * embedding + 2, history_hidden), nn.ReLU())\\n        self.head = nn.Sequential(nn.Linear(hidden + history_hidden, hidden), nn.ReLU(),\\n                                  nn.Linear(hidden, ACTION_COUNT))\\n\\n    def forward(self, batch):\\n        boards, metadata, old_boards, old_meta, owners = batch\\n        current = self.current(torch.cat((self.pieces(boards).flatten(1), metadata), dim=1))\\n        old = self.history(torch.cat((self.pieces(old_boards).flatten(1), old_meta), dim=1))\\n        # Each complete repetition map is pooled as an unordered set.\\n        pooled = old.new_zeros((len(boards), old.shape[1])).index_add(0, owners, old)\\n        count = torch.bincount(owners, minlength=len(boards)).clamp_min(1).sqrt().unsqueeze(1)\\n        return self.head(torch.cat((current, pooled / count), dim=1))\\n\\n\\n@torch.no_grad()\\ndef greedy_action(model, state, legal_ids, device):\\n    if not legal_ids:\\n        raise ValueError(\\"No legal action is available.\\")\\n    q = model(batch_states([state], device))[0]\\n    legal = torch.tensor(legal_ids, dtype=torch.long, device=device)\\n    return int(legal[q[legal].argmax()].item())\\n\\n\\nclass Policy:\\n    \\"\\"\\"Load an exported checkpoint once and serve the website\'s Python protocol.\\"\\"\\"\\n    def __init__(self, checkpoint=None, device=\\"cpu\\"):\\n        if not checkpoint:\\n            raise ValueError(\\"A trained DQN checkpoint is required.\\")\\n        torch.set_num_threads(2)\\n        self.device = torch.device(device)\\n        saved = torch.load(checkpoint, map_location=\\"cpu\\", weights_only=True)\\n        if saved.get(\\"schema\\") != MODEL_SCHEMA:\\n            raise ValueError(\\"Unsupported DQN checkpoint format.\\")\\n        self.model = QNetwork(**saved[\\"network_config\\"]).to(self.device)\\n        self.model.load_state_dict(saved[\\"model\\"])\\n        self.model.eval()\\n\\n    def select_move(self, observation, time_budget_ms=800):\\n        return {\\"action_id\\": greedy_action(self.model, encode_state(observation),\\n                                           observation[\\"legal_action_ids\\"], self.device)}\\n", "dqn_training.py": "\\"\\"\\"Fixed-opponent DQN and Double DQN with validation and early stopping.\\"\\"\\"\\nfrom collections import deque\\nfrom dataclasses import asdict, dataclass\\nfrom datetime import datetime, timezone\\nimport csv\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport random\\nimport sys\\nimport time\\nimport zipfile\\n\\nimport torch\\nfrom torch import nn\\n\\nfrom dqn_model import MODEL_SCHEMA, QNetwork, batch_states, encode_state, greedy_action\\nfrom rules import Game, PRESETS, action_id, action_move\\n\\nENDGAMES = {p[\\"id\\"]: p for p in PRESETS if p[\\"id\\"] in (\\"rook\\", \\"cannon\\", \\"horse\\")}\\n\\n\\nclass FixedOpponentEnv:\\n    \\"\\"\\"One learner decision and (if needed) a fixed-policy opponent reply.\\"\\"\\"\\n    def __init__(self, max_plies=160, learner=\\"red\\", opponent=None):\\n        if learner not in (\\"red\\", \\"black\\") or type(max_plies) is not int or not 4 <= max_plies <= 1000:\\n            raise ValueError(\\"Use red/black and a ply limit from 4 to 1000.\\")\\n        self.max_plies, self.learner = max_plies, learner\\n        self.opponent = opponent\\n        self.game = None\\n\\n    def _reply(self):\\n        move = (self.opponent(self.game.observation(), self.rng) if self.opponent\\n                else self.rng.choice(self.game.legal))\\n        self.game.step(move, actor=\\"fixed-opponent\\")\\n\\n    def reset(self, fen, seed=0):\\n        self.rng = random.Random(seed)\\n        self.game = Game(fen, self.max_plies)\\n        opening = int(self.game.side != self.learner)\\n        # Align the limit to the learner\'s next turn, so a time-limit transition\\n        # can bootstrap from a learner state. Never exceed the requested cap.\\n        self.game.max_plies = self.max_plies - ((self.max_plies - opening) % 2)\\n        if opening and self.game.outcome() is None:\\n            self._reply()\\n        return self.game.observation(), {\\"outcome\\": self.game.outcome()}\\n\\n    def step(self, action):\\n        if self.game is None or self.game.outcome() is not None:\\n            raise ValueError(\\"Reset before moving in a new episode.\\")\\n        if self.game.side != self.learner:\\n            raise RuntimeError(\\"A training step must start on the learner\'s turn.\\")\\n        before = len(self.game.history)\\n        move = action_move(action) if type(action) is int else action\\n        self.game.step(move, actor=\\"dqn\\")\\n        if self.game.outcome() is None:\\n            self._reply()\\n        outcome = self.game.outcome()\\n        winner = outcome[\\"winner\\"] if outcome else None\\n        reward = (1.0 if winner == self.learner else -1.0) if winner else 0.0\\n        return (self.game.observation(), reward,\\n                bool(outcome and outcome[\\"terminated\\"]), bool(outcome and outcome[\\"truncated\\"]),\\n                {\\"outcome\\": outcome, \\"learner\\": self.learner,\\n                 \\"environment_plies\\": len(self.game.history) - before})\\n\\n\\n@dataclass\\nclass TrainConfig:\\n    preset: str = \\"rook\\"\\n    learner: str = \\"red\\"\\n    min_episodes: int = 1000\\n    max_episodes: int = 3000\\n    max_plies: int = 160\\n    eval_every: int = 100\\n    eval_games: int = 50\\n    patience: int = 5\\n    min_delta: float = 0.02\\n    seed: int = 42\\n    learning_rate: float = 0.0003\\n    gamma: float = 0.99\\n    replay_size: int = 10000\\n    batch_size: int = 64\\n    warmup_steps: int = 512\\n    train_every: int = 4\\n    target_every: int = 250\\n    epsilon_start: float = 1.0\\n    epsilon_end: float = 0.1\\n    epsilon_decay_steps: int = 20000\\n    device: str = \\"auto\\"\\n    algorithm: str = \\"dqn\\"\\n\\n    def validate(self):\\n        if self.algorithm not in (\\"dqn\\", \\"double_dqn\\"):\\n            raise ValueError(\\"Choose dqn or double_dqn.\\")\\n        if self.preset not in ENDGAMES or self.learner not in (\\"red\\", \\"black\\"):\\n            raise ValueError(\\"Unknown preset or learner colour.\\")\\n        for name in (\\"min_episodes\\", \\"max_episodes\\", \\"max_plies\\", \\"eval_every\\", \\"eval_games\\",\\n                     \\"patience\\", \\"replay_size\\", \\"batch_size\\", \\"warmup_steps\\", \\"train_every\\",\\n                     \\"target_every\\", \\"epsilon_decay_steps\\"):\\n            if type(getattr(self, name)) is not int or getattr(self, name) < 1:\\n                raise ValueError(name + \\" must be a positive integer.\\")\\n        if self.max_episodes < self.min_episodes:\\n            raise ValueError(\\"MAX_EPISODES must be at least MIN_EPISODES.\\")\\n        if not 4 <= self.max_plies <= 1000 or self.replay_size < self.batch_size:\\n            raise ValueError(\\"Invalid ply limit or replay size.\\")\\n        if not 0 <= self.epsilon_end <= self.epsilon_start <= 1 or not 0 <= self.gamma <= 1:\\n            raise ValueError(\\"Invalid epsilon or discount factor.\\")\\n        if not math.isfinite(self.learning_rate) or self.learning_rate <= 0:\\n            raise ValueError(\\"Learning rate must be positive and finite.\\")\\n        if not math.isfinite(self.min_delta) or self.min_delta < 0:\\n            raise ValueError(\\"Minimum improvement must be finite and non-negative.\\")\\n\\n\\n@dataclass\\nclass EarlyStopping:\\n    min_episodes: int\\n    patience: int\\n    min_delta: float\\n    best_score: float = -math.inf\\n    reference_score: float = -math.inf\\n    bad_evals: int = 0\\n    monitoring_started: bool = False\\n\\n    def update(self, episode, score):\\n        if not math.isfinite(score):\\n            raise ValueError(\\"Validation score must be finite.\\")\\n        improved = score > self.best_score\\n        if improved:\\n            self.best_score = score\\n        if episode < self.min_episodes:\\n            self.bad_evals = 0\\n        elif not self.monitoring_started:\\n            self.monitoring_started = True\\n            self.reference_score = self.best_score\\n            self.bad_evals = 0\\n        elif score > self.reference_score + self.min_delta:\\n            self.reference_score = score\\n            self.bad_evals = 0\\n        else:\\n            self.bad_evals += 1\\n        return improved, self.monitoring_started and self.bad_evals >= self.patience\\n\\n\\n@torch.no_grad()\\ndef legal_next_values(target, next_states, legal_ids, terminated, device, online=None):\\n    \\"\\"\\"Double DQN selects with online and evaluates with target, using legal moves.\\"\\"\\"\\n    values = torch.zeros(len(next_states), device=device)\\n    live = [i for i, terminal in enumerate(terminated) if not terminal]\\n    if live:\\n        batch = batch_states([next_states[i] for i in live], device)\\n        scores = target(batch)\\n        selection = online(batch) if online is not None else scores\\n        for row, index in enumerate(live):\\n            if not legal_ids[index]:\\n                raise ValueError(\\"A nonterminal transition needs legal next actions.\\")\\n            legal = torch.tensor(legal_ids[index], dtype=torch.long, device=device)\\n            chosen = legal[selection[row, legal].argmax()]\\n            values[index] = scores[row, chosen]\\n    return values\\n\\n\\ndef optimize(model, target, optimizer, replay, rng, config, device):\\n    batch = rng.sample(list(replay), config.batch_size)\\n    states, actions, rewards, next_states, terminals, next_ids = zip(*batch)\\n    predicted = model(batch_states(states, device)).gather(\\n        1, torch.tensor(actions, device=device).unsqueeze(1)).squeeze(1)\\n    expected = torch.tensor(rewards, device=device) + config.gamma * legal_next_values(\\n        target, next_states, next_ids, terminals, device,\\n        online=model if config.algorithm == \\"double_dqn\\" else None)\\n    loss = nn.functional.smooth_l1_loss(predicted, expected)\\n    if not torch.isfinite(loss):\\n        raise FloatingPointError(\\"Non-finite DQN loss; the run was stopped.\\")\\n    optimizer.zero_grad(set_to_none=True)\\n    loss.backward()\\n    nn.utils.clip_grad_norm_(model.parameters(), 10.0)\\n    optimizer.step()\\n    return float(loss.item())\\n\\n\\n@torch.no_grad()\\ndef evaluate(model, config, device, split=\\"validation\\", games=None, positions=None):\\n    \\"\\"\\"Fixed seed bank, fresh envs, no exploration or training-state mutation.\\"\\"\\"\\n    count = games or config.eval_games\\n    if positions is not None and (not positions or count % len(positions)):\\n        raise ValueError(\\"Validation games must cover every board equally.\\")\\n    stats = {\\"wins\\": 0, \\"losses\\": 0, \\"draws\\": 0, \\"truncations\\": 0, \\"environment_plies\\": 0}\\n    per_board = {p[\\"id\\"]: {key: 0 for key in (\\"wins\\", \\"losses\\", \\"draws\\", \\"truncations\\")} for p in positions or []}\\n    previous_mode = model.training if model is not None else False\\n    if model is not None:\\n        model.eval()\\n    try:\\n        for index in range(count):\\n            env = FixedOpponentEnv(config.max_plies, config.learner)\\n            position = positions[index % len(positions)] if positions is not None else ENDGAMES[config.preset]\\n            seed = (f\\"{split}:{position[\'id\']}:opponent:{config.seed}:{index // len(positions)}\\"\\n                    if positions is not None else f\\"{split}:opponent:{config.seed}:{index}\\")\\n            obs, info = env.reset(position[\\"fen\\"], seed)\\n            policy_rng = random.Random(f\\"{split}:random-policy:{config.seed}:{index}\\")\\n            while info[\\"outcome\\"] is None:\\n                action = (policy_rng.choice(obs[\\"legal_action_ids\\"]) if model is None\\n                          else greedy_action(model, encode_state(obs), obs[\\"legal_action_ids\\"], device))\\n                obs, reward, terminated, truncated, info = env.step(action)\\n            result = info[\\"outcome\\"]\\n            key = (\\"truncations\\" if result[\\"truncated\\"] else \\"draws\\" if result[\\"winner\\"] is None\\n                   else \\"wins\\" if result[\\"winner\\"] == config.learner else \\"losses\\")\\n            stats[key] += 1\\n            if positions is not None:\\n                per_board[position[\\"id\\"]][key] += 1\\n            stats[\\"environment_plies\\"] += len(env.game.history)\\n    finally:\\n        if model is not None:\\n            model.train(previous_mode)\\n    details = {\\"per_board\\": per_board} if positions is not None else {}\\n    return {**stats, **details, \\"games\\": count, \\"split\\": split,\\n            \\"score\\": (stats[\\"wins\\"] - stats[\\"losses\\"]) / count,\\n            \\"win_rate\\": stats[\\"wins\\"] / count}\\n\\n\\ndef _write_json(path, value):\\n    temporary = path.with_suffix(path.suffix + \\".tmp\\")\\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    temporary.replace(path)\\n\\n\\ndef _save_torch(path, value):\\n    temporary = path.with_suffix(path.suffix + \\".tmp\\")\\n    torch.save(value, temporary)\\n    temporary.replace(path)\\n\\n\\ndef _append_csv(path, row):\\n    exists = path.is_file()\\n    with path.open(\\"a\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\n        writer = csv.DictWriter(stream, fieldnames=list(row))\\n        if not exists:\\n            writer.writeheader()\\n        writer.writerow(row)\\n\\n\\ndef train(config, run_dir, resume=False, source_hashes=None, board_bank=None, on_checkpoint=None):\\n    \\"\\"\\"Run explicitly. No training starts when this module is imported.\\"\\"\\"\\n    config.validate()\\n    if board_bank is not None and config.eval_games % len(board_bank.validation):\\n        raise ValueError(\\"Validation games must cover every board equally.\\")\\n    validation_positions = board_bank.validation if board_bank is not None else None\\n    run_dir = Path(run_dir).resolve()\\n    if resume:\\n        if not (run_dir / \\"last.pt\\").is_file():\\n            raise FileNotFoundError(\\"No last.pt checkpoint in this run folder.\\")\\n    elif run_dir.exists() and any(run_dir.iterdir()):\\n        raise ValueError(\\"Use a new run folder, or explicitly resume an existing run.\\")\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    torch.set_num_threads(2)\\n    device = torch.device(\\"cuda\\" if config.device == \\"auto\\" and torch.cuda.is_available()\\n                          else \\"cpu\\" if config.device == \\"auto\\" else config.device)\\n    torch.manual_seed(config.seed)\\n    rng = random.Random(config.seed)\\n    replay_rng = random.Random(config.seed + 1)\\n    model, target = QNetwork().to(device), QNetwork().to(device)\\n    target.load_state_dict(model.state_dict())\\n    target.eval()\\n    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)\\n    replay = deque(maxlen=config.replay_size)\\n    stopper = EarlyStopping(config.min_episodes, config.patience, config.min_delta)\\n    steps = updates = total_plies = completed = 0\\n    best_episode = None\\n    last_loss = None\\n    last_validation = None\\n    if resume:\\n        saved = torch.load(run_dir / \\"last.pt\\", map_location=\\"cpu\\", weights_only=True)\\n        if saved.get(\\"schema\\") != MODEL_SCHEMA or saved.get(\\"kind\\") != \\"training\\":\\n            raise ValueError(\\"Unsupported training checkpoint.\\")\\n        for key, value in asdict(config).items():\\n            previous = saved[\\"config\\"].get(key, \\"dqn\\" if key == \\"algorithm\\" else None)\\n            if key not in (\\"max_episodes\\", \\"device\\") and previous != value:\\n                raise ValueError(\\"Resume configuration differs: \\" + key)\\n        if saved.get(\\"source_hashes\\", {}) != (source_hashes or {}):\\n            raise ValueError(\\"Resume requires the same source snapshot.\\")\\n        if saved.get(\\"board_bank_sha256\\") != (board_bank.sha256 if board_bank is not None else None):\\n            raise ValueError(\\"Resume requires the same training and validation board bank.\\")\\n        model.load_state_dict(saved[\\"model\\"])\\n        target.load_state_dict(saved[\\"target\\"])\\n        optimizer.load_state_dict(saved[\\"optimizer\\"])\\n        for state in optimizer.state.values():\\n            for key, value in state.items():\\n                if isinstance(value, torch.Tensor):\\n                    state[key] = value.to(device)\\n        replay.extend(saved[\\"replay\\"])\\n        rng.setstate(saved[\\"rng\\"])\\n        replay_rng.setstate(saved[\\"replay_rng\\"])\\n        torch.set_rng_state(saved[\\"torch_rng\\"])\\n        if device.type == \\"cuda\\" and saved.get(\\"cuda_rng\\"):\\n            torch.cuda.set_rng_state_all(saved[\\"cuda_rng\\"])\\n        stopper = EarlyStopping(**saved[\\"early_stopping\\"])\\n        completed, steps, updates, total_plies = (saved[k] for k in (\\"episode\\", \\"steps\\", \\"updates\\", \\"total_plies\\"))\\n        best_episode, last_loss, last_validation = saved[\\"best_episode\\"], saved[\\"last_loss\\"], saved[\\"validation\\"]\\n        if completed >= config.max_episodes:\\n            raise ValueError(\\"The saved run already reached MAX_EPISODES. Increase it to resume.\\")\\n        if not (run_dir / \\"best.pt\\").is_file():\\n            raise FileNotFoundError(\\"Resume needs the original best.pt beside last.pt.\\")\\n        # CSV rows written after the last completed checkpoint may be from an\\n        # interrupted episode range; remove those rows before replaying that range.\\n        for name in (\\"episodes.csv\\", \\"validation.csv\\"):\\n            path = run_dir / name\\n            if path.exists():\\n                with path.open(newline=\\"\\", encoding=\\"utf-8\\") as stream:\\n                    reader = csv.DictReader(stream)\\n                    fields, rows = reader.fieldnames, list(reader)\\n                with path.open(\\"w\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\n                    writer = csv.DictWriter(stream, fieldnames=fields)\\n                    writer.writeheader()\\n                    writer.writerows(row for row in rows if int(row[\\"episode\\"]) <= completed)\\n    else:\\n        print(\\"Checking the random-policy baseline...\\", flush=True)\\n        _write_json(run_dir / \\"baseline.json\\", evaluate(None, config, device, positions=validation_positions))\\n\\n    if board_bank is not None:\\n        _write_json(run_dir / \\"board_bank.json\\", board_bank.data)\\n\\n    _write_json(run_dir / \\"config.json\\", {**asdict(config), \\"actual_device\\": str(device),\\n                \\"source_hashes\\": source_hashes or {}, \\"torch_version\\": str(torch.__version__),\\n                \\"python_version\\": sys.version.split()[0], \\"opponent\\": \\"uniform-legal-random-v1\\",\\n                \\"board_bank_sha256\\": board_bank.sha256 if board_bank is not None else None,\\n                \\"validation_scope\\": \\"separate validation starting boards; fixed seeds; no test-board selection\\"\\n                if board_bank is not None else \\"same preset; separate fixed random seeds; no unseen-board claim\\"})\\n    started = time.monotonic()\\n    stop_reason = \\"max_episodes\\"\\n\\n    def checkpoint(kind, validation):\\n        payload = {\\"schema\\": MODEL_SCHEMA, \\"kind\\": kind, \\"network_config\\": model.config,\\n                   \\"model\\": model.state_dict(), \\"config\\": asdict(config), \\"episode\\": completed,\\n                   \\"steps\\": steps, \\"updates\\": updates, \\"total_plies\\": total_plies,\\n                   \\"validation\\": validation, \\"source_hashes\\": source_hashes or {}}\\n        if board_bank is not None:\\n            payload[\\"board_bank_sha256\\"] = board_bank.sha256\\n        if kind == \\"training\\":\\n            payload.update(target=target.state_dict(), optimizer=optimizer.state_dict(), replay=list(replay),\\n                           rng=rng.getstate(), replay_rng=replay_rng.getstate(), torch_rng=torch.get_rng_state(),\\n                           cuda_rng=torch.cuda.get_rng_state_all() if device.type == \\"cuda\\" else [],\\n                           early_stopping=asdict(stopper), best_episode=best_episode, last_loss=last_loss)\\n        return payload\\n\\n    print(f\\"Algorithm: {config.algorithm} | Device: {device} | At least {config.min_episodes} training episodes | Folder: {run_dir}\\", flush=True)\\n    print(\\"Early-stop patience starts at the first validation after the minimum episode budget.\\", flush=True)\\n    try:\\n        for episode in range(completed + 1, config.max_episodes + 1):\\n            env = FixedOpponentEnv(config.max_plies, config.learner)\\n            position = board_bank.training_position(episode, config.seed) if board_bank is not None else ENDGAMES[config.preset]\\n            obs, info = env.reset(position[\\"fen\\"], f\\"train:opponent:{config.seed}:{episode}\\")\\n            if info[\\"outcome\\"] is not None:\\n                raise ValueError(\\"The training preset ended before the learner could move.\\")\\n            total_plies += len(env.game.history)\\n            episode_steps, reward, epsilon = 0, 0.0, config.epsilon_start\\n            while True:\\n                state = encode_state(obs)\\n                fraction = min(1.0, steps / config.epsilon_decay_steps)\\n                epsilon = config.epsilon_start + fraction * (config.epsilon_end - config.epsilon_start)\\n                action = (rng.choice(obs[\\"legal_action_ids\\"]) if rng.random() < epsilon else\\n                          greedy_action(model, state, obs[\\"legal_action_ids\\"], device))\\n                next_obs, reward, terminated, truncated, info = env.step(action)\\n                next_state = encode_state(next_obs)\\n                # A truncated game retains next-state legal IDs for bootstrapping.\\n                replay.append((state, action, reward, next_state, terminated,\\n                               tuple(next_obs[\\"legal_action_ids\\"]) if not terminated else ()))\\n                steps += 1\\n                episode_steps += 1\\n                total_plies += info[\\"environment_plies\\"]\\n                if steps >= config.warmup_steps and len(replay) >= config.batch_size and steps % config.train_every == 0:\\n                    last_loss = optimize(model, target, optimizer, replay, replay_rng, config, device)\\n                    updates += 1\\n                    if updates % config.target_every == 0:\\n                        target.load_state_dict(model.state_dict())\\n                obs = next_obs\\n                if terminated or truncated:\\n                    break\\n            completed = episode\\n            board_details = {\\"board_id\\": position[\\"id\\"], \\"initial_fen\\": position[\\"fen\\"]} if board_bank is not None else {}\\n            _append_csv(run_dir / \\"episodes.csv\\", {\\"episode\\": episode, \\"learner_steps\\": steps,\\n                        \\"environment_plies\\": total_plies, \\"episode_plies\\": len(env.game.history),\\n                        \\"reward\\": reward, \\"reason\\": info[\\"outcome\\"][\\"reason\\"],\\n                        \\"epsilon\\": epsilon, \\"loss\\": last_loss, \\"updates\\": updates, **board_details})\\n            if episode % 25 == 0:\\n                print(f\\"Episode {episode}/{config.max_episodes} | steps {steps} | epsilon {epsilon:.3f}\\", flush=True)\\n            if episode % config.eval_every == 0 or episode == config.min_episodes or episode == config.max_episodes:\\n                last_validation = evaluate(model, config, device, positions=validation_positions)\\n                improved, stop = stopper.update(episode, last_validation[\\"score\\"])\\n                if improved:\\n                    best_episode = episode\\n                    _save_torch(run_dir / \\"best.pt\\", checkpoint(\\"policy\\", last_validation))\\n                _append_csv(run_dir / \\"validation.csv\\", {\\"episode\\": episode,\\n                            **{k: v for k, v in last_validation.items() if k != \\"per_board\\"},\\n                            \\"bad_evals\\": stopper.bad_evals, \\"best_episode\\": best_episode})\\n                if board_bank is not None:\\n                    _write_json(run_dir / f\\"validation_ep{episode:04d}.json\\", last_validation)\\n                _write_json(run_dir / \\"example_game.json\\", env.game.export())\\n                _save_torch(run_dir / \\"last.pt\\", checkpoint(\\"training\\", last_validation))\\n                if on_checkpoint is not None:\\n                    on_checkpoint(run_dir, improved)\\n                print(f\\"Validation {episode}: W/L/D/T = {last_validation[\'wins\']}/{last_validation[\'losses\']}/\\"\\n                      f\\"{last_validation[\'draws\']}/{last_validation[\'truncations\']} | score {last_validation[\'score\']:.3f} | \\"\\n                      f\\"patience {stopper.bad_evals}/{config.patience}\\", flush=True)\\n                if stop:\\n                    stop_reason = \\"early_stopping\\"\\n                    break\\n    except KeyboardInterrupt:\\n        # last.pt refers only to a completed validation checkpoint. Preserve it;\\n        # do not label a partially played episode as completed training.\\n        stop_reason = \\"interrupted\\"\\n        print(\\"Interrupted. Resume from the latest completed last.pt checkpoint.\\", flush=True)\\n    summary = {\\"algorithm\\": config.algorithm,\\n               \\"episodes_completed\\": completed, \\"minimum_reached\\": completed >= config.min_episodes,\\n               \\"learner_steps\\": steps, \\"environment_plies\\": total_plies, \\"updates\\": updates,\\n               \\"stop_reason\\": stop_reason, \\"best_episode\\": best_episode,\\n               \\"best_validation_score\\": stopper.best_score if math.isfinite(stopper.best_score) else None,\\n               \\"seconds_this_session\\": time.monotonic() - started, \\"run_dir\\": str(run_dir)}\\n    _write_json(run_dir / \\"summary.json\\", summary)\\n    if best_episode is not None:\\n        best = torch.load(run_dir / \\"best.pt\\", map_location=device, weights_only=True)\\n        model.load_state_dict(best[\\"model\\"])\\n    print(json.dumps(summary, indent=2), flush=True)\\n    return {\\"model\\": model, \\"config\\": config, \\"device\\": device, \\"summary\\": summary, \\"run_dir\\": run_dir}\\n\\n\\ndef export_policy(run_dir, model_id=None, model_name=None):\\n    \\"\\"\\"Create a complete model folder ZIP for the website, without replay memory.\\"\\"\\"\\n    run_dir = Path(run_dir)\\n    saved = torch.load(run_dir / \\"best.pt\\", map_location=\\"cpu\\", weights_only=True)\\n    preset = saved[\\"config\\"][\\"preset\\"]\\n    algorithm = saved[\\"config\\"].get(\\"algorithm\\", \\"dqn\\")\\n    label = \\"Double DQN\\" if algorithm == \\"double_dqn\\" else \\"DQN\\"\\n    folder = model_id if model_id is not None else f\\"{algorithm}_{preset}\\"\\n    if not folder or any(c not in \\"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-\\" for c in folder):\\n        raise ValueError(\\"Use a simple portable model ID.\\")\\n    output = run_dir / f\\"{folder}_model.zip\\"\\n    manifest = {\\"schema\\": \\"xiangqi-lab.model.v1\\", \\"id\\": model_id or f\\"custom-{algorithm.replace(\'_\', \'-\')}-{preset}\\",\\n                \\"name\\": model_name or f\\"{label} {preset}\\", \\"description\\": f\\"Best validation checkpoint, episode {saved[\'episode\']}; trained as {saved[\'config\'][\'learner\']}.\\",\\n                \\"protocol\\": \\"python\\", \\"entrypoint\\": \\"policy.py\\", \\"python\\": \\"{python}\\",\\n                \\"checkpoint\\": \\"best.pt\\", \\"device\\": \\"cpu\\", \\"timeout_seconds\\": 60}\\n    readme = (f\\"# {label} {preset}\\\\n\\\\nTrained as {saved[\'config\'][\'learner\']}, selected at episode {saved[\'episode\']}.\\\\n\\"\\n              \\"Extract this folder under xiangqi_lab/models/. Install requirements.txt in the Python environment \\"\\n              \\"that starts the website, or set model.json\'s python path to a separate environment with PyTorch. \\"\\n              \\"The portable app\'s bundled Python does not include PyTorch.\\\\n\\\\n\\"\\n              \\"Use Models > Refresh list > Test before selecting the model. Legal moves are masked for either \\"\\n              \\"colour; playing strength for the untrained colour and unseen boards is not established.\\\\n\\\\n\\"\\n              \\"Validation uses the same fixed starting board with separate opponent seeds. \\"\\n              \\"It is used for checkpoint selection, not a held-out-board generalisation claim.\\\\n\\")\\n    if saved.get(\\"board_bank_sha256\\"):\\n        readme = readme.replace(\\"Validation uses the same fixed starting board with separate opponent seeds.\\",\\n                                \\"Validation uses separate starting boards and fixed opponent seeds; test boards are excluded.\\")\\n    with zipfile.ZipFile(output, \\"w\\", zipfile.ZIP_DEFLATED) as archive:\\n        archive.write(run_dir / \\"best.pt\\", f\\"{folder}/best.pt\\")\\n        archive.write(Path(__file__).with_name(\\"dqn_model.py\\"), f\\"{folder}/dqn_model.py\\")\\n        archive.writestr(f\\"{folder}/policy.py\\", \\"from dqn_model import Policy\\\\n\\")\\n        archive.writestr(f\\"{folder}/model.json\\", json.dumps(manifest, indent=2))\\n        archive.writestr(f\\"{folder}/requirements.txt\\", \\"torch>=2.6,<3\\\\n\\")\\n        archive.writestr(f\\"{folder}/README.md\\", readme)\\n        for name in (\\"config.json\\", \\"summary.json\\", \\"baseline.json\\", \\"episodes.csv\\", \\"validation.csv\\", \\"example_game.json\\", \\"board_bank.json\\"):\\n            if (run_dir / name).exists():\\n                archive.write(run_dir / name, f\\"{folder}/training_report/{name}\\")\\n    return output\\n"}')
SOURCE_HASHES = {'rules.py': '8fc7f6b38578e0d2538c546d8a755a61f787e5e0c5640e4d025e7a362d9c7225', 'training.py': '1575c584754c0516beafa332210eab8249a709b427fd7a351e31aa8e226429d3', 'notebook_tools.py': '552844feeba9103769325966e9ab852f71969bb9a19a28b3076add7d6122e7b0', 'dqn_model.py': '4f34c3a3a5687c9ced81330315ddfde5dd13cd65f8631ec4b520dd7ac139bb78', 'dqn_training.py': '827260f673469d6b61b5165fe71b9c8672bd32c9e068b75f4a827be21c5acc6f'}
SOURCE_BASE_COMMIT = 'c1ef171cbb88c3e682b900e912ca83e02c9bfcc4'
SNAPSHOT_ID = hashlib.sha256(json.dumps(SOURCE_HASHES, sort_keys=True).encode()).hexdigest()[:16]
SOURCE_DIR = Path(tempfile.gettempdir()) / "xiangqi_lab_colab" / SNAPSHOT_ID
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in SOURCE_FILES.items():
    data = source.encode("utf-8")
    if hashlib.sha256(data).hexdigest() != SOURCE_HASHES[name]:
        raise ValueError("Source check failed: " + name)
    path = SOURCE_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError("Snapshot was edited on disk. Restart with a clean runtime.")
    if not path.exists():
        path.write_bytes(data)
    loaded = sys.modules.get(path.stem)
    if loaded is not None and Path(getattr(loaded, "__file__", "")).resolve() != path.resolve():
        raise RuntimeError("Another rule version is loaded. Restart the runtime, then run all cells.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()
from rules import Game, action_id, action_move
from training import XiangqiEnv
from notebook_tools import ENDGAMES, action_mask, board_svg, replay_episode

OUTPUT_DIR = Path.cwd() / "xiangqi_output"
OUTPUT_DIR.mkdir(exist_ok=True)
try:
    from IPython.display import SVG, display
except ImportError:
    SVG = None  # A normal Python process can still run all rule checks.

def show_board(game, title="Xiangqi Lab"):
    svg = board_svg(game, title)
    path = OUTPUT_DIR / "board.svg"
    path.write_text(svg, encoding="utf-8")
    if SVG is not None:
        display(SVG(data=svg))
    else:
        print("Board saved:", path)

print("Rules ready. Snapshot:", SNAPSHOT_ID)
print("Project base commit:", SOURCE_BASE_COMMIT)
print("Rules: xiangqi-research-v1 | Episode format: xiangqi-lab.episode.v1")

## 2. Check the environment

Checks legal moves, rewards, game endings, and reset for the three endgames.

In [ ]:
def check_environment():
    for key, preset in ENDGAMES.items():
        probe = XiangqiEnv(max_plies=20)
        state, _ = probe.reset(fen=preset["fen"])
        assert probe.game.outcome() is None and state["legal_moves"]
        assert [len(state["board_planes"]), len(state["board_planes"][0]),
                len(state["board_planes"][0][0])] == [14, 10, 9]
        assert sum(action_mask(probe)) == len(state["legal_moves"])
        for move, index in zip(state["legal_moves"], state["legal_action_ids"]):
            assert action_move(index) == move and action_id(move) == index
        before = probe.game.export()
        try:
            probe.step("a0a0")
        except ValueError:
            pass
        else:
            raise AssertionError("An illegal move was accepted")
        assert probe.game.export() == before
        next_state, reward, ended, limited, info = probe.step(state["legal_action_ids"][0])
        assert next_state["side_to_move"] != state["side_to_move"]
        assert info["acting_side"] == state["side_to_move"]
        print("PASS:", key, "— legal actions, move, and invalid move check")

    wins = [
        ("red", "4k4/3R1R3/9/9/9/9/9/9/9/3K5 w - - 0 1", "d8e8"),
        ("black", "3k5/9/9/9/9/9/9/9/3r1r3/4K4 b - - 0 1", "d1e1"),
    ]
    for winner, fen, move in wins:
        probe.reset(fen=fen)
        _, reward, ended, limited, info = probe.step(move)
        assert ended and not limited and reward == 1.0
        assert info["outcome"]["winner"] == winner
        assert info["rewards"][winner] == 1 and sum(info["rewards"].values()) == 0
        assert not any(action_mask(probe))
        print("PASS:", winner, "win — winner +1, loser -1")

    probe = XiangqiEnv(max_plies=2)
    probe.reset(fen=ENDGAMES["rook"]["fen"])
    for _ in range(2):
        _, reward, ended, limited, info = probe.step(probe.game.legal[0])
    assert limited and not ended and reward == 0 and not any(action_mask(probe))
    try:
        probe.step(probe.game.legal[0])
    except ValueError:
        pass
    else:
        raise AssertionError("An ended game accepted another move")
    probe.reset(fen=ENDGAMES["rook"]["fen"])
    assert probe.game.history == [] and probe.game.outcome() is None
    assert len(probe.game.repetitions) == 1 and sum(probe.game.repetitions.values()) == 1
    print("PASS: move limit, stop, and reset")
    print("All environment checks passed. No model was trained.")

check_environment()

## 3. Choose a board

Resets the chosen endgame and displays the board, FEN, and legal moves.

In [ ]:
PRESET_ID = "rook"  # @param ["rook", "cannon", "horse"]
MAX_PLIES = 160     # @param {type:"integer"}
SEED = 42          # @param {type:"integer"}

if PRESET_ID not in ENDGAMES:
    raise ValueError("Choose rook, cannon or horse.")
if type(MAX_PLIES) is not int or not 2 <= MAX_PLIES <= 1000:
    raise ValueError("MAX_PLIES must be an integer from 2 to 1000.")
for key, item in ENDGAMES.items():
    board = Game(item["fen"])
    print(f"{key:7} | {item['name']:22} | {sum(bool(p) for row in board.board for p in row)} pieces")

env = XiangqiEnv(max_plies=MAX_PLIES)
obs, reset_info = env.reset(fen=ENDGAMES[PRESET_ID]["fen"], seed=SEED)
rng = random.Random(SEED)  # The demo policy owns its random generator.
show_board(env.game, ENDGAMES[PRESET_ID]["name"])
print("Start FEN:", obs["fen"])
print("Legal moves:", obs["legal_moves"])

## 4. View the state

Shows the board data format, current player, and legal move IDs.

In [ ]:
obs = env.game.observation()
mask = action_mask(env)
print("Board shape: (14, 10, 9)")
print("Piece order:", obs["plane_order"])
print("Turn:", obs["side_to_move"])
print("Allowed actions:", sum(mask), "of", len(mask))
print("Examples:", list(zip(obs["legal_moves"], obs["legal_action_ids"]))[:8])
print("End state:", env.game.outcome())

## 5. Make one move

Makes one legal move and shows the updated board, reward, and game status.

In [ ]:
MOVE = ""  # @param {type:"string"}
if env.game.outcome() is not None:
    print("Game ended. Run 'Choose a board' to reset.")
else:
    selected_move = MOVE.strip() or env.game.legal[0]
    obs, reward, terminated, truncated, info = env.step(selected_move)
    print("Move:", selected_move, "| Acting side:", info["acting_side"])
    print("Reward:", reward, "| Rewards by colour:", info["rewards"])
    print("Terminated:", terminated, "| Truncated:", truncated)
    print("Result:", info["outcome"])
    show_board(env.game, ENDGAMES[PRESET_ID]["name"])

## 6. Short automatic demo

Plays up to `DEMO_PLIES` random legal moves and prints the move log.

In [ ]:
DEMO_PLIES = 6  # @param {type:"integer"}
if type(DEMO_PLIES) is not int or not 0 <= DEMO_PLIES <= 20:
    raise ValueError("Use 0 to 20 plies for this short demo.")
for _ in range(DEMO_PLIES):
    if env.game.outcome() is not None:
        break
    selected_move = rng.choice(env.game.legal)
    obs, reward, terminated, truncated, info = env.step(selected_move)
    env.game.history[-1]["actor"] = "colab-random-demo"
    print(f"{len(env.game.history):3} | {info['acting_side']:5} | {selected_move} | reward {reward:+.0f}")
    if terminated or truncated:
        break
show_board(env.game, ENDGAMES[PRESET_ID]["name"])
print("Result:", env.game.outcome() or "Game still in progress")
print("No learning or weight update took place.")

## 7. Save a game for the website

Replays the game to check it, then saves a JSON file for the website.
Downloads the file when `DOWNLOAD_JSON` is `True`.

In [ ]:
DOWNLOAD_JSON = False  # @param {type:"boolean"}
episode = env.game.export()
episode["run_metadata"] = {
    "source": "colab-environment-demo", "trained": False,
    "preset_id": PRESET_ID, "seed": SEED,
    "source_base_commit": SOURCE_BASE_COMMIT, "source_sha256": SOURCE_HASHES,
}
replayed = replay_episode(episode)
assert replayed.fen == env.game.fen
assert replayed.outcome() == env.game.outcome()
assert replayed.repetitions == env.game.repetitions
# Keep previous exports when this cell is run again.
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
episode_path = OUTPUT_DIR / f"{PRESET_ID}-{stamp}.json"
episode_path.write_text(json.dumps(episode, indent=2, ensure_ascii=False), encoding="utf-8")
print("Replay check passed.")
print("Saved:", episode_path, "| Moves:", len(episode["moves"]))
if DOWNLOAD_JSON:
    try:
        from google.colab import files
    except ImportError:
        print("Outside Colab: copy the saved file from the path above.")
    else:
        files.download(str(episode_path))

## 8. Load a game

Loads and checks a game from `IMPORT_PATH`, then displays its board.
An empty path skips this cell.

In [ ]:
IMPORT_PATH = ""  # @param {type:"string"}
if IMPORT_PATH.strip():
    imported_episode = json.loads(Path(IMPORT_PATH).read_text(encoding="utf-8-sig"))
    imported_game = replay_episode(imported_episode)
    show_board(imported_game, "Game from the website")
    print("Moves:", len(imported_game.history), "| Result:", imported_game.outcome())
else:
    print("Skipped. The environment setup is complete.")

## 9. Load DQN

Loads the DQN network, training loop, and model export tools.

In [ ]:
import torch
from dqn_training import TrainConfig, train, export_policy
print("PyTorch:", torch.__version__)
print("Training device:", "cuda" if torch.cuda.is_available() else "cpu")

## 10. Set training options

Sets at least 1,000 training episodes and the validation rules for early stopping.

In [ ]:
ALGORITHM = "double_dqn"  # @param ["dqn", "double_dqn"]
TRAIN_PRESET = "rook"  # @param ["rook", "cannon", "horse"]
MIN_EPISODES = 1000    # @param {type:"integer"}
MAX_EPISODES = 3000    # @param {type:"integer"}
EVAL_EVERY = 100       # @param {type:"integer"}
EVAL_GAMES = 50        # @param {type:"integer"}
PATIENCE = 5           # @param {type:"integer"}
MIN_DELTA = 0.02       # @param {type:"number"}
TRAIN_SEED = 42        # @param {type:"integer"}
EPSILON_DECAY_STEPS = 100000  # @param {type:"integer"}

if MIN_EPISODES < 1000:
    raise ValueError("Use at least 1,000 training episodes.")
train_config = TrainConfig(
    preset=TRAIN_PRESET, learner="red", min_episodes=MIN_EPISODES,
    max_episodes=MAX_EPISODES, eval_every=EVAL_EVERY,
    eval_games=EVAL_GAMES, patience=PATIENCE,
    min_delta=MIN_DELTA, seed=TRAIN_SEED,
    epsilon_decay_steps=EPSILON_DECAY_STEPS,
    algorithm=ALGORITHM,
)
train_config.validate()
print("Algorithm:", ALGORITHM)
print("Train:", TRAIN_PRESET, "| Learner: Red | Opponent: random legal moves")
print("Episode budget:", MIN_EPISODES, "to", MAX_EPISODES)
print("Exploration: 1.0 to 0.1 over", EPSILON_DECAY_STEPS, "learner decisions.")
print("After the minimum, stop after", PATIENCE, "validation checks without a clear gain.")
print("Validation uses separate seeds on the same starting board.")

## 11. Train the model

Starts training when `START_TRAINING` is checked, saves checkpoints, and stops on the validation rule.
`RESUME_FOLDER` resumes an existing run.

In [ ]:
START_TRAINING = False  # @param {type:"boolean"}
RESUME_FOLDER = ""      # @param {type:"string"}

training_result = None
if START_TRAINING:
    from datetime import datetime, timezone
    if RESUME_FOLDER.strip():
        run_folder = Path(RESUME_FOLDER).expanduser().resolve()
    else:
        run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        run_folder = OUTPUT_DIR / "runs" / (train_config.algorithm + "-" + TRAIN_PRESET + "-" + run_name)
    training_result = train(
        train_config, run_folder, resume=bool(RESUME_FOLDER.strip()),
        source_hashes=SOURCE_HASHES,
    )
else:
    print("Training is off. No training episodes have been run by this cell.")

## 12. Export the best model

Packs the best checkpoint, policy code, and results into a ZIP for the website.
Downloads it when `DOWNLOAD_MODEL` is checked.

In [ ]:
SAVED_RUN_FOLDER = ""   # @param {type:"string"}
DOWNLOAD_MODEL = False  # @param {type:"boolean"}

saved_run = (Path(SAVED_RUN_FOLDER).expanduser().resolve() if SAVED_RUN_FOLDER.strip()
             else training_result["run_dir"] if training_result else None)
if saved_run is not None:
    model_zip = export_policy(saved_run)
    print("Model package:", model_zip)
    print("The website's model interpreter needs PyTorch. See the README inside the ZIP.")
    if DOWNLOAD_MODEL:
        try:
            from google.colab import files
        except ImportError:
            print("Outside Colab: copy the ZIP from the path above.")
        else:
            files.download(str(model_zip))
else:
    print("No training run selected. Train first or set SAVED_RUN_FOLDER.")

## 13. Compare with the last DQN run

Compares win rates and recent variation with the saved 100,000-step-exploration DQN run.
Saves a table and chart after training finishes.

In [ ]:
# Compare this run with the saved DQN run; do not update either model.
import csv
import json
from pathlib import Path
import statistics

reference_rates = [1.0, 0.04, 0.0, 0.02, 0.0, 0.0, 0.06, 0.0, 0.0, 0.0, 0.0, 0.04, 0.0, 0.0, 0.08]
reference_episodes = list(range(100, 1501, 100))
reference_config = {
    "preset": "rook", "learner": "red", "min_episodes": 1000, "max_episodes": 3000,
    "max_plies": 160, "eval_every": 100, "eval_games": 50, "patience": 5,
    "min_delta": 0.02, "seed": 42, "learning_rate": 0.0003, "gamma": 0.99,
    "replay_size": 10000, "batch_size": 64, "warmup_steps": 512,
    "train_every": 4, "target_every": 250, "epsilon_start": 1.0,
    "epsilon_end": 0.1, "epsilon_decay_steps": 100000, "device": "auto",
}

if not training_result or training_result["config"].algorithm != "double_dqn":
    print("No Double DQN run to compare.")
else:
    comparison_folder = Path(training_result["run_dir"])
    current_config = json.loads((comparison_folder / "config.json").read_text())
    changed = {key: [value, current_config.get(key)] for key, value in reference_config.items()
               if current_config.get(key) != value}
    if changed:
        raise ValueError("Comparison settings differ: " + str(changed))
    with (comparison_folder / "validation.csv").open() as stream:
        comparison_rows = list(csv.DictReader(stream))
    if not comparison_rows:
        raise ValueError("No validation results yet.")
    current_rates = [float(row["win_rate"]) for row in comparison_rows]
    current_episodes = [int(row["episode"]) for row in comparison_rows]

    def rate_summary(rates, episodes, steps, best_episode):
        recent = rates[-5:]
        return {"best_checkpoint_win_rate": rates[episodes.index(best_episode)],
                "final_win_rate": rates[-1], "recent_mean_win_rate": statistics.mean(recent),
                "recent_std_win_rate": statistics.pstdev(recent), "recent_win_rates": recent,
                "recent_episodes": episodes[-5:], "training_episodes": episodes[-1],
                "learner_steps": steps, "best_episode": best_episode}

    comparison = {
        "reference_run": "rook-20261021T135047973987Z",
        "current_run": comparison_folder.name,
        "scope": "Same fixed rook board and random opponent; validation, not independent testing.",
        "dqn": rate_summary(reference_rates, reference_episodes, 82473, 100),
        "double_dqn": rate_summary(current_rates, current_episodes,
                                   training_result["summary"]["learner_steps"],
                                   training_result["summary"]["best_episode"]),
    }
    print("Metric                         DQN        Double DQN")
    for key, label in [("best_checkpoint_win_rate", "Best checkpoint win rate"),
                       ("final_win_rate", "Final win rate"),
                       ("recent_mean_win_rate", "Last 5 checks: mean win rate"),
                       ("recent_std_win_rate", "Last 5 checks: spread (SD)")]:
        print(f"{label:<30} {comparison['dqn'][key]:>8.1%} {comparison['double_dqn'][key]:>12.1%}")
    print("DQN recent wins:", comparison["dqn"]["recent_win_rates"])
    print("Double DQN recent wins:", comparison["double_dqn"]["recent_win_rates"])
    print("Less variation alone does not mean stronger play. Compare win rates too.")
    print(comparison["scope"])
    (comparison_folder / "comparison.json").write_text(json.dumps(comparison, indent=2))
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Chart skipped: matplotlib is not installed.")
    else:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(reference_episodes, [100*x for x in reference_rates], "o-", label="DQN")
        ax.plot(current_episodes, [100*x for x in current_rates], "o-", label="Double DQN")
        ax.set(xlabel="Training episodes", ylabel="Validation win rate (%)", ylim=(-2, 102),
               title="Fixed rook endgame vs random opponent (50 games per check)")
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(comparison_folder / "comparison.png", dpi=160)
        plt.show()
    print("Comparison saved in:", comparison_folder)